In [0]:
# =============================================================================
# NOTEBOOK  : bronze_pos_transactions_stream
# PURPOSE   : Ingest POS stream → bronze.pos_transactions
# LAYER     : Bronze (raw ingestion — no transformation)
# FREQUENCY : Continuous (availableNow in dev, micro-batch in prod)
# FORMAT    : JSONL (dev) / Kafka topic (prod)
# NOTE      : To switch to Kafka — set USE_KAFKA = True in Cell 1
#             and fill in KAFKA_BOOTSTRAP_SERVERS + KAFKA_TOPIC in config.json
#             Everything downstream (audit, write, quality check) stays identical
# =============================================================================
 
# ── Imports & Config ──────────────────────────────────────────────────
import sys, json
sys.path.insert(0, "/Workspace/Shared/mini_project_a3t7")
 
from utils.audit import start_audit, end_audit
from utils.quality_checks import assert_not_empty
from utils.schema_registry import BRONZE_POS_TRANSACTIONS_SCHEMA
 
from pyspark.sql.functions import current_timestamp, lit, col, to_date, from_json
 
with open("/Workspace/Shared/mini_project_a3t7/config/config.json") as f:
    cfg = json.load(f)
 
# ── SWITCH THIS TO True WHEN KAFKA IS READY ───────────────────────────────────
USE_KAFKA = False
 
TARGET_TABLE = cfg["tables"]["bronze_pos_transactions"]
CHECKPOINT   = cfg["checkpoint_paths"]["bronze_pos_transactions_stream"]
PIPELINE     = "bronze_pos_transactions_stream"

In [0]:
# ── Audit + Read + Write ─────────────────────────────────────────────
run_id = start_audit(spark, PIPELINE, TARGET_TABLE)
 
try:
    # ── READ ──────────────────────────────────────────────────────────────────
    if not USE_KAFKA:
        # ── DEV: Read from JSONL file in landing volume ───────────────────────
        # Simulates real-time stream using static file
        # availableNow trigger used — processes file and stops
        # When switching to Kafka, only this block changes
        SOURCE_PATH = cfg["landing_paths"]["pos_transactions_stream"]
 
        df = (
            spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "json")
            .option("multiLine", "false")
            .option("cloudFiles.schemaLocation", CHECKPOINT + "/schema")
            .schema(BRONZE_POS_TRANSACTIONS_SCHEMA)
            .load(SOURCE_PATH)
        )
 
        # source_file from metadata — actual file path
        df = df.withColumn("source_file", col("_metadata.file_path"))
 
    else:
        # ── PROD: Read from Kafka topic ───────────────────────────────────────
        # Kafka delivers each message as a binary `value` field
        # We parse it as JSON using the same schema
        # No cloudFiles, no schemaLocation needed for Kafka
        KAFKA_SERVERS = cfg["kafka"]["bootstrap_servers"]  
        KAFKA_TOPIC   = cfg["kafka"]["pos_topic"]          
 
        raw_df = (
            spark.readStream
            .format("kafka")
            .option("kafka.bootstrap.servers", KAFKA_SERVERS)
            .option("subscribe", KAFKA_TOPIC)
            .option("startingOffsets", "latest")
            .load()
        )
 
        # Kafka value is binary — cast to string then parse as JSON
        df = (
            raw_df
            .select(
                from_json(
                    col("value").cast("string"),
                    BRONZE_POS_TRANSACTIONS_SCHEMA
                ).alias("data")
            )
            .select("data.*")
        )
 
        # No source file for Kafka — null
        df = df.withColumn("source_file", lit(f"kafka://{KAFKA_TOPIC}"))
 
    # ── ADD AUDIT COLUMNS ─────────────────────────────────────────────────────
    # _source_file already added above per branch
    # ingested_at and pipeline_run_id uniform across all tables
    df = (
        df
        .withColumn("_source",         lit("pos_realtime_stream"))
        .withColumn("ingested_at",     current_timestamp())
        .withColumn("ingested_date",   to_date(current_timestamp()).cast("string"))
        .withColumn("pipeline_run_id", lit(run_id))
    )
 
    # ── WRITE ─────────────────────────────────────────────────────────────────
    # trigger(availableNow=True) for dev (file-based)
    # Remove trigger() entirely for Kafka — runs as continuous micro-batch
    write_stream = (
        df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT)
        .partitionBy("ingested_date")
    )
 
    if not USE_KAFKA:
        query = write_stream.trigger(availableNow=True).toTable(TARGET_TABLE)
    else:
        # No trigger — continuous micro-batch for Kafka
        query = write_stream.toTable(TARGET_TABLE)
 
    query.awaitTermination()
 
    # ── QUALITY CHECK ─────────────────────────────────────────────────────────
    written_df = (
        spark.read.table(TARGET_TABLE)
        .where(f"pipeline_run_id = '{run_id}'")
    )
    rows_written = written_df.count()
    assert_not_empty(written_df, PIPELINE)
 
    print(f"[WRITE] {rows_written} rows written to {TARGET_TABLE}")
 
    # ── END AUDIT (SUCCESS) ───────────────────────────────────────────────────
    end_audit(spark, run_id, "SUCCESS", rows_written=rows_written)
 
except Exception as e:
    end_audit(spark, run_id, "FAILED", error=str(e))
    raise